# 🚀 Supreme Polymath AI Grandmaster - Google Colab Training Notebook
## Fine-Tune Super AI Model on Free T4 GPU using Unsloth + QLoRA

**Instructions:**
1. In Google Colab menu, select **Runtime -> Change runtime type**.
2. Select **T4 GPU** under Hardware accelerator, then click **Save**.
3. Run all cells sequentially below to train your custom Polymath AI model.

In [ ]:
# 1. Install Unsloth, PyTorch, & dependencies optimized for Colab T4 GPU
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes datasets triton
print("✅ Unsloth and Training dependencies installed successfully!")

In [ ]:
# 2. Load Foundation Model in 4-bit (Qwen 2.5 7B Instruct)
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048 # Max context length
dtype = None # Auto detection (Float16 / Bfloat16)
load_in_4bit = True # 4-bit quantization to fit in Free Colab T4 GPU

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print("✅ Model & Tokenizer loaded into T4 GPU VRAM!")

In [ ]:
# 3. Add QLoRA Adapters to Model
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # 30% less VRAM
    random_state = 3407,
)
print("✅ QLoRA Adapters successfully attached!")

In [ ]:
# 4. Prepare Polymath AI Educational Dataset
from datasets import Dataset

polymath_dataset = [
    {
        "instruction": "ពន្យល់ពីគោលការណ៍គ្រឹះរបស់ Artificial Intelligence (AI) សម្រាប់អ្នកទើបចាប់ផ្តើម",
        "output": "បញ្ញាសិប្បនិម្មិត (AI) គឺជាបច្ចេកវិទ្យាកុំព្យូទ័រដែលត្រូវបានរចនាឡើងដើម្បីត្រាប់តាមសមត្ថភាពគិត វិភាគ និងរៀនសូត្ររបស់មនុស្ស។ AI ដំណើរការដោយប្រើប្រាស់ Algorithm និង Data ដើម្បីស្វែងរករូបមន្ត (Patterns) និងធ្វើការសន្និដ្ឋាន。\n\n📘 គោលការណ៍គ្រឹះទាំង ៣៖\n១. Data Ingestion: ការបញ្ចូលទិន្នន័យ\n២. Pattern Recognition: ការស្វែងរករូបមន្តក្នុងទិន្នន័យ\n៣. Continuous Learning: ការរៀនសូត្រស្វ័យប្រវត្តិដើម្បីអភិវឌ្ឍចម្លើយឱ្យកាន់តែច្បាស់លាស់។"
    },
    {
        "instruction": "Explain the first-principles intuition of Transformer Architecture in Deep Learning.",
        "output": "The Transformer architecture fundamentally revolutionized AI by replacing sequential processing (RNNs) with Parallel Attention Mechanisms.\n\nKey Mechanism - Self-Attention:\nInstead of reading word-by-word sequentially, Self-Attention allows every token to calculate mathematical relationships (Query, Key, Value vectors) with all other tokens in a sentence simultaneously.\n\nWhy this matters:\n1. Massive parallel computation on GPUs.\n2. Long-range context capturing without forgetting early tokens."
    },
    {
        "instruction": "តើអ្វីទៅជា Prompt Engineering ហើយតើវាមានសារៈសំខាន់យ៉ាងណា?",
        "output": "Prompt Engineering គឺជាសិល្បៈនិងវិទ្យាសាស្ត្រនៃការសរសេរពាក្យបញ្ជា (Instructions) ទៅកាន់ AI ដើម្បីឱ្យវាឆ្លើយតបមកវិញបានត្រឹមត្រូវ ច្បាស់លាស់ និងមានប្រសិទ្ធភាពបំផុត។\n\n🎯 បច្ចេកទេសសំខាន់ៗ៖\n១. Define Role: ប្រាប់ AI ឱ្យដើរតួជាអ្នកជំនាញ (ឧទាហរណ៍៖ អ្នកជាគ្រូបង្រៀន AI)\n២. Provide Context: ផ្តល់ព័ត៌មានផ្ទៃខាងក្រោយឱ្យច្បាស់\n៣. Specify Output Format: ប្រាប់ទម្រង់ចម្លើយដែលចង់បាន (ឧទាហរណ៍៖ ជាតារាង ឬ Bullet points)"
    }
]

polymath_prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}"""

def format_prompts(examples):
    texts = []
    for inst, out in zip(examples["instruction"], examples["output"]):
        text = polymath_prompt.format(inst, out) + "<eos>"
        texts.append(text)
    return {"text": texts}

dataset = Dataset.from_list(polymath_dataset)
dataset = dataset.map(format_prompts, batched = True)
print("✅ Polymath Educational Dataset formatted & ready for training!")

In [ ]:
# 5. Configure & Start Unsloth SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("🚀 Starting Polymath AI Model Fine-Tuning...")
trainer_stats = trainer.train()
print("🎉 Fine-Tuning Completed Successfully!")

In [ ]:
# 6. Test Model Inference
FastLanguageModel.for_inference(model)
inputs = tokenizer(
[
    polymath_prompt.format(
        "ពន្យល់ពីគោលការណ៍គ្រឹះរបស់ Artificial Intelligence (AI) សម្រាប់អ្នកទើបចាប់ផ្តើម",
        "",
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
print(tokenizer.batch_decode(outputs)[0])

In [ ]:
# 7. Save & Export Trained Model to GGUF format
model.save_pretrained_gguf("polymath_ai_model", tokenizer, quantization_method = "q4_k_m")
print("🎉 Super Polymath AI Model exported to 'polymath_ai_model' GGUF format!")